In [1]:
import pandas as pd
from utils import convert_tags, generate_pairs, generate_answer, generate_graph, remove_implicit_tags

# Benchmark Construction

This notebook takes in the prepared user data and food data and output the benchmark `NutriGraphQA_benchmark.csv` we proposed in the paper. Specifically:

 - We first establish the rubric to create the candidate user food pairs.

 - Then, we create the questions and the corresponding answers. 
 
 - Finally, we construct the graph served as knowledge base. 
 
Note that our benchmark have a different level of difficulty for both *question settings* and *task settings*. For the question settings:  

 - **Easy:** An easy question is a one-on-one match or contradict. 

 - **Medium:** A medium question is a multi-on-multi match or contradict.

 - **Hard:** A hard question is a mixed combination of match and contradict. 

For the task settings: 

 - **Easy:** The answer is directly healthy or not with no additional requests. (Binary Classification).  

 - **Medium:** Collection of nutrition tags. The answers are the list of nutritional tags. (Multi-label Classification)

 - **Hard:** The natural text generation. The answers are a coherent answer. (Text Generation)

In [2]:
users_w_tags = pd.read_csv('../processed_data/user_tagging.csv').drop_duplicates()
food_list = pd.read_csv('../processed_data/reduced_mixed_dishes_v4.csv')
fndds_ingredients = pd.read_csv('../data/fndds_ingredients.csv')
user_habits = pd.read_csv('../processed_data/trimmedHabit.csv')

In [3]:
# Manually define the tags we involve in this work.
user_primary_nutrition_tags = ['low_carb', 'low_sugar', 'low_sodium', 'low_cholesterol',
    'low_saturated_fat', 'low_calorie', 'high_calorie', 'low_protein', 'high_protein' ]

# These tags are the status and conditions. 
auxiliary_tags = ['Weight loss/Low calorie diet', 'Low fat/Low cholesterol diet', 'Low salt/Low sodium diet',
    'Sugar free/Low sugar diet', 'Diabetic diet', 'Weight gain/Muscle building diet', 'Low carbohydrate diet',
    'High protein diet', 'Renal/Kidney diet', 'opioid_misuse', 'hypertension', 'diabetes', 'obesity'
]

food_primary_nutrition_tags = [
    'low_carb', 'high_carb',
    'low_sugar', 'high_sugar',
    'low_sodium', 'high_sodium',
    'low_calorie', 'high_calorie',
    'low_protein', 'high_protein',
    'low_cholesterol', 'high_cholesterol',
    'low_saturated_fat', 'high_saturated_fat',
]

# The mapping between the status and the nutrients it needs.
reference_dict = {
    'obesity': ['low_calorie'], 
    'opioid_misuse': ['high_protein', 'low_sugar', 'low_sodium'],
    'hypertension': ['low_sodium'], 
    'diabetes': ['low_sugar', 'low_carb'], 
    'Weight loss/Low calorie diet': ['low_calorie'], 
    'Low fat/Low cholesterol diet': ['low_cholesterol', 'low_saturated_fat'], 
    'Low salt/Low sodium diet': ['low_sodium'], 
    'Sugar free/Low sugar diet': ['low_sugar'], 
    'Diabetic diet': ['low_sugar', 'low_carb'],  
    'Weight gain/Muscle building diet': ['high_calorie', 'high_protein'], 
    'Low carbohydrate diet': ['low_carb'], 
    'High protein diet': ['high_protein'], 
    'Renal/Kidney diet': ['low_protein']}

### Step 1: User Food Pair Generation

In [4]:
# Filter out the users who have no tags.
user_list = []

for _, row in users_w_tags.iterrows():
    # Check for auxiliary tags; if none, skip this user.
    if not any(row[keyword] for keyword in auxiliary_tags):
        continue

    # Filter out implicitly associated nutrition tags
    row = remove_implicit_tags(row, reference_dict)
    
    # Initialize the user's data with user_id
    user_nutrition_dict = {'user_id': row['SEQN']}
    user_nutrition_dict = convert_tags(row, user_primary_nutrition_tags, user_nutrition_dict)

    # Only add the user if they have at least one nutrition tag
    if len(user_nutrition_dict) > 1:
        user_list.append(user_nutrition_dict)

# Potential user list for the next step. 
print(len(user_list))

30188


In [5]:
from tqdm import tqdm


easy_count, medium_count, hard_count = 10, 5, 2
pair_list = []

for _, row in tqdm(food_list.iterrows(), total=food_list.shape[0]):
    # Convert high/low tags to uniform tags
    food_nutrition_dict = convert_tags(row, food_primary_nutrition_tags, {})
    nutrition_list = list(food_nutrition_dict.keys())
    # The mapping is based on each food - Each food has a list of users. 
    pair_dict = {'food_id': row['food_id'], 'food_tag': food_nutrition_dict,
        'easy': [], 'medium': [], 'hard': []
    }
    
    # Generate pairs for each difficulty level
    pair_dict['easy'] = generate_pairs(food_nutrition_dict, nutrition_list, user_list, easy_count, 'easy', pair_dict['food_id'])
    if len(nutrition_list) > 1:
        pair_dict['medium'] = generate_pairs(food_nutrition_dict, nutrition_list, user_list, medium_count, 'medium', pair_dict['food_id'])
        pair_dict['hard'] = generate_pairs(food_nutrition_dict, nutrition_list, user_list, hard_count, 'hard', pair_dict['food_id'])
        
    pair_list.append(pair_dict)

  0%|          | 0/662 [00:00<?, ?it/s]

100%|██████████| 662/662 [07:51<00:00,  1.40it/s]


A quick check on the number of pairs generated.

In [6]:
easy_sum = medium_sum = hard_sum = 0

for food_user_dict in pair_list:
    easy_sum += len(food_user_dict['easy'])
    medium_sum += len(food_user_dict['medium'])
    hard_sum += len(food_user_dict['hard'])

print(easy_sum, medium_sum, hard_sum)

6620 2836 1310


### Step 2 & 3: Create the question, answer, and the graph as context. 

In [7]:
NutriGraphQA_benchmark = [['difficulty', 'question_easy', 'answer_easy', 'question_medium', 'answer_medium', 'question_hard', 'answer_hard', 'node_list', 'edge_list']]
for food_user_pairs in pair_list:
    food_id = food_user_pairs['food_id']
    food_tag = food_user_pairs['food_tag']

    # Do note that we have two difficult settings for the KGQA benchmark - The question level and the task level. 
    question_levels = ['easy', 'medium', 'hard']
    for level in question_levels:
        for user in food_user_pairs[level]:
            user_id = user['user_id']

            question_base = 'Based on the nutrients the food provides and the user needs, please answer '
            # generate questions and answers
            question_easy = question_base + 'if the food {} is a healthy option to the user {}? Please answer with yes or no.'.format(food_id, user_id)
            question_medium = question_base + 'what nutrient tags are used when judging if food {} for is healthy or unhealthy to user {}.'.format(food_id, user_id)
            question_hard = question_base + 'if the food {} is a healthy option to the user {}? Please answer with a short sentence.'.format(food_id, user_id)
            answer_easy, answer_medium, answer_hard = generate_answer(food_tag, user)
            
            # genearte graph
            node_list, edge_list = generate_graph(food_id, user_id, food_list, fndds_ingredients, users_w_tags, user_habits, food_primary_nutrition_tags, reference_dict)
            # Note that the level is the question level, not the task level.
            NutriGraphQA_benchmark.append([level, question_easy, answer_easy, question_medium, answer_medium, question_hard, answer_hard, node_list, edge_list])

df = pd.DataFrame(NutriGraphQA_benchmark[1:], columns=NutriGraphQA_benchmark[0])

This is the final output of our constructed benchmark.

In [8]:
df.to_csv('../processed_data/NutriGraphQA_benchmark.csv', index=False)
df.head()

,difficulty,question_easy,answer_easy,question_medium,answer_medium,question_hard,answer_hard,node_list,edge_list
0,easy,Based on the nutrients the food provides and t...,No,Based on the nutrients the food provides and t...,high_cholesterol,Based on the nutrients the food provides and t...,"No, because the food is high in cholesterol.","[[1, {'name': 'Adobo, with noodles', 'attr': 5...","[[1, belongs to, 2], [1, has, 3], [1, has, 4],..."
1,easy,Based on the nutrients the food provides and t...,Yes,Based on the nutrients the food provides and t...,low_sugar,Based on the nutrients the food provides and t...,"Yes, because the food is low in sugar.","[[1, {'name': 'Adobo, with noodles', 'attr': 5...","[[1, belongs to, 2], [1, has, 3], [1, has, 4],..."
2,easy,Based on the nutrients the food provides and t...,Yes,Based on the nutrients the food provides and t...,low_sugar,Based on the nutrients the food provides and t...,"Yes, because the food is low in sugar.","[[1, {'name': 'Adobo, with noodles', 'attr': 5...","[[1, belongs to, 2], [1, has, 3], [1, has, 4],..."
3,easy,Based on the nutrients the food provides and t...,Yes,Based on the nutrients the food provides and t...,high_protein,Based on the nutrients the food provides and t...,"Yes, because the food is high in protein.","[[1, {'name': 'Adobo, with noodles', 'attr': 5...","[[1, belongs to, 2], [1, has, 3], [1, has, 4],..."
4,easy,Based on the nutrients the food provides and t...,No,Based on the nutrients the food provides and t...,high_cholesterol,Based on the nutrients the food provides and t...,"No, because the food is high in cholesterol.","[[1, {'name': 'Adobo, with noodles', 'attr': 5...","[[1, belongs to, 2], [1, has, 3], [1, has, 4],..."
